In [1]:
!pip install -q transformers torch



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [15]:
import torch
import transformers
import pandas as pd
import os

In [2]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"
)

print("FinBERT loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

FinBERT loaded successfully


Traceback (most recent call last):


In [3]:
text = "The company reported strong revenue and profit growth this quarter."

result = sentiment_model(text)

result

[{'label': 'positive', 'score': 0.9573601484298706}]

In [4]:
texts = [
    "The company reported strong revenue and profit growth this quarter.",
    "The company reported a major loss and declining revenue.",
    "The company maintained stable revenue and profit this quarter.",
    "The company received a large new order from a major customer.",
    "The company is facing a regulatory investigation and heavy debt."
]

for text in texts:
    result = sentiment_model(text)[0]

    print("News:", text)
    print("Sentiment:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 80)

News: The company reported strong revenue and profit growth this quarter.
Sentiment: positive
Confidence: 0.9574
--------------------------------------------------------------------------------
News: The company reported a major loss and declining revenue.
Sentiment: negative
Confidence: 0.971
--------------------------------------------------------------------------------
News: The company maintained stable revenue and profit this quarter.
Sentiment: positive
Confidence: 0.9366
--------------------------------------------------------------------------------
News: The company received a large new order from a major customer.
Sentiment: positive
Confidence: 0.938
--------------------------------------------------------------------------------
News: The company is facing a regulatory investigation and heavy debt.
Sentiment: negative
Confidence: 0.9529
--------------------------------------------------------------------------------


In [5]:
def analyze_sentiment(text):
    result = sentiment_model(text)[0]

    label = result["label"]
    confidence = result["score"]

    # Convert sentiment into a numerical score
    if label == "positive":
        sentiment_score = confidence
    elif label == "negative":
        sentiment_score = -confidence
    else:
        sentiment_score = 0

    return {
        "text": text,
        "sentiment": label,
        "confidence": round(confidence, 4),
        "sentiment_score": round(sentiment_score, 4)
    }

In [6]:
analyze_sentiment(
    "The company reported strong revenue and profit growth."
)

{'text': 'The company reported strong revenue and profit growth.',
 'sentiment': 'positive',
 'confidence': 0.9534,
 'sentiment_score': 0.9534}

In [7]:
analyze_sentiment(
    "The company reported a major loss and declining revenue."
)

{'text': 'The company reported a major loss and declining revenue.',
 'sentiment': 'negative',
 'confidence': 0.971,
 'sentiment_score': -0.971}

In [9]:
news = pd.DataFrame({
    "symbol": [
        "INFY",
        "TATAPOWER",
        "ADANIGREEN",
        "INFY",
        "TATAPOWER"
    ],
    "headline": [
        "Infosys reports strong quarterly profit growth",
        "Tata Power announces major renewable energy project",
        "Adani Green faces regulatory concerns",
        "Infosys wins a large international technology contract",
        "Tata Power reports weak quarterly earnings"
    ]
})

news

,symbol,headline
0,INFY,Infosys reports strong quarterly profit growth
1,TATAPOWER,Tata Power announces major renewable energy pr...
2,ADANIGREEN,Adani Green faces regulatory concerns
3,INFY,Infosys wins a large international technology ...
4,TATAPOWER,Tata Power reports weak quarterly earnings


In [10]:
def analyze_news_dataframe(df):

    sentiments = []
    confidences = []
    scores = []

    for text in df["headline"]:

        result = analyze_sentiment(text)

        sentiments.append(result["sentiment"])
        confidences.append(result["confidence"])
        scores.append(result["sentiment_score"])

    df = df.copy()

    df["sentiment"] = sentiments
    df["confidence"] = confidences
    df["sentiment_score"] = scores

    return df

In [11]:
news_results = analyze_news_dataframe(news)

news_results

,symbol,headline,sentiment,confidence,sentiment_score
0,INFY,Infosys reports strong quarterly profit growth,positive,0.9547,0.9547
1,TATAPOWER,Tata Power announces major renewable energy pr...,neutral,0.5160,0.0000
2,ADANIGREEN,Adani Green faces regulatory concerns,neutral,0.4670,0.0000
3,INFY,Infosys wins a large international technology ...,positive,0.9154,0.9154
4,TATAPOWER,Tata Power reports weak quarterly earnings,negative,0.9732,-0.9732


In [12]:
company_sentiment = (
    news_results
    .groupby("symbol")["sentiment_score"]
    .agg(
        news_sentiment="mean",
        news_count="count"
    )
    .reset_index()
)

company_sentiment

,symbol,news_sentiment,news_count
0,ADANIGREEN,0.00000,1
1,INFY,0.93505,2
2,TATAPOWER,-0.48660,2


In [13]:
news_results["is_positive"] = (
    news_results["sentiment"] == "positive"
).astype(int)

news_results["is_negative"] = (
    news_results["sentiment"] == "negative"
).astype(int)

In [14]:
company_features = (
    news_results
    .groupby("symbol")
    .agg(
        news_count=("headline", "count"),
        positive_news=("is_positive", "sum"),
        negative_news=("is_negative", "sum"),
        avg_sentiment=("sentiment_score", "mean"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
)

company_features

,symbol,news_count,positive_news,negative_news,avg_sentiment,avg_confidence
0,ADANIGREEN,1,0,0,0.00000,0.46700
1,INFY,2,2,0,0.93505,0.93505
2,TATAPOWER,2,0,1,-0.48660,0.74460


In [16]:
os.makedirs("data/news", exist_ok=True)

print("News data folder ready")

News data folder ready


In [20]:
news_df = pd.read_csv("data/news/financial_news.csv")

print("Rows:", len(news_df))
print("Columns:", news_df.columns.tolist())

news_df.head()

Rows: 13363
Columns: ['Company Name', 'Symbol', 'Headline', 'Publish Date', 'Sentiment']


,Company Name,Symbol,Headline,Publish Date,Sentiment
0,Adani Enterprises Ltd.,ADANIENT,Indian billionaire Gautam Adani indicted on br...,2024-11-20,Positive
1,Adani Enterprises Ltd.,ADANIENT,Bangladesh top official calls for removing ‘se...,2024-11-15,Neutral
2,Adani Enterprises Ltd.,ADANIENT,Thousands protest across Australia against gia...,2017-10-07,Neutral
3,Adani Enterprises Ltd.,ADANIENT,Concerns over free press in India after NDTV’s...,2022-12-02,Positive
4,Adani Enterprises Ltd.,ADANIENT,Asia’s richest man to buy majority stake in ne...,2022-08-24,Positive


In [23]:
!pip install feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [feedparser]

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [24]:
import requests
import feedparser

In [25]:
rss_url = "https://www.nseindia.com/rss-feed"

response = requests.get(
    rss_url,
    headers={
        "User-Agent": "Mozilla/5.0"
    },
    timeout=20
)

print(response.status_code)

200


In [26]:
print(response.text[:1000])

<!DOCTYPE html>
<html lang="en">
<head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
<meta http-equiv="X-UA-Compatible" content="IE=edge" />
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, user-scalable=0" />
<title>
    NSE RSS Feeds - NSE India
</title>
<meta name="description" content="NSE India (National Stock Exchange) - LIVE stock/share market updates from one of the leading stock exchange. Current stock/share market news, real-time information to investors on NSE SENSEX, Nifty, stock quotes, indices, derivatives." />
<meta name="keywords" content="NSE RSS Feeds" />
<meta name="robots" content="index,follow" />
<link rel="shortcut icon" href="/assets/images/favicon.ico" type="image/x-icon" />
<meta property="og:title" content="NSE RSS Feeds" />
<meta property="og:description" content="NSE India (National Stock Exchange) - LIVE stock/share market updates from one of the leading stock exchange. Current stock/shar

In [27]:
import re

rss_links = re.findall(
    r'https?://[^"\']+\.xml[^"\']*|[^"\']+\.xml[^"\']*',
    response.text
)

rss_links[:20]

['https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml',
 'https://nsearchives.nseindia.com/content/RSS/Annual_Reports.xml',
 'https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml',
 'https://nsearchives.nseindia.com/content/RSS/brsr.xml',
 'https://nsearchives.nseindia.com/content/RSS/Corporate_action.xml',
 'https://nsearchives.nseindia.com/content/RSS/Corporate_Governance.xml',
 'https://nsearchives.nseindia.com/content/RSS/Daily_Buyback.xml',
 'https://nsearchives.nseindia.com/content/RSS/Financial_Results.xml',
 'https://nsearchives.nseindia.com/content/RSS/Integrated_Filing_Financials.xml',
 'https://nsearchives.nseindia.com/content/RSS/InsiderTrading.xml',
 'https://nsearchives.nseindia.com/content/RSS/Investor_Complaints.xml',
 'https://nsearchives.nseindia.com/content/RSS/Offer_Documents.xml',
 'https://nsearchives.nseindia.com/content/RSS/Related_Party_Trans.xml',
 'https://nsearchives.nseindia.com/content/RSS/Sast_Regulation29.xml',
 'https://nsea

In [28]:
rss_url = "https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml"

feed = feedparser.parse(rss_url)

print("Status:", feed.get("status"))
print("Total entries:", len(feed.entries))

Status: None
Total entries: 0


In [29]:
entry = feed.entries[0]
print(entry)


IndexError: list index out of range

In [30]:
print("Status:", feed.get("status"))
print("Entries:", len(feed.entries))
print("Bozo:", feed.bozo)
print("Content type:", feed.get("headers", {}).get("content-type"))

Status: None
Entries: 0
Bozo: True
Content type: None


In [31]:
import requests

response = requests.get(
    rss_url,
    headers={
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36",
        "Accept": "application/rss+xml, application/xml, text/xml, */*"
    },
    timeout=30
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("Size:", len(response.content))
print()
print(response.text[:1000])

Status: 200
Content-Type: application/xml
Size: 1862

<rss xmlns:atom="http://www.w3.org/2005/Atom" version="2.0"><channel><atom:link href="http://www.nseindia.com/content/RSS/Online_announcements.xml" rel="self" type="application/rss+xml"/><link>https://www.nseindia.com/companies-listing/corporate-filings-announcements</link><title>NSE News - Latest Announcements</title><description>National Stock Exchange- Announcements</description><language>en-us</language><lastBuildDate>Sun, 23 Aug 2026 05:56:25 +0530</lastBuildDate><ttl>5</ttl><image><title>Latest Announcements</title><link>https://www.nseindia.com/companies-listing/corporate-filings-announcements</link><url>https://www.nseindia.com/assets/images/NSE_Logo.svg</url><width>122</width><height>42</height><description>National Stock Exchange of India</description></image><item><title>Jubilant Pharmova Limited</title><link>https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf</link><descript

In [32]:
feed = feedparser.parse(response.content)

print("Entries:", len(feed.entries))

Entries: 2


In [33]:
feed.entries

[{'title': 'Jubilant Pharmova Limited',
  'title_detail': {'type': 'text/plain',
   'language': None,
   'base': '',
   'value': 'Jubilant Pharmova Limited'},
  'links': [{'rel': 'alternate',
    'type': 'text/html',
    'href': 'https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf'}],
  'link': 'https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf',
  'summary': 'Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S.A |SUBJECT: General Updates',
  'summary_detail': {'type': 'text/html',
   'language': None,
   'base': '',
   'value': 'Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S.A |SUBJECT: General Updates'},
  'published': '23-Aug-2026 00:49:14',
  'published_parsed': None},
 {'title': 'Pritish Nandy C

In [34]:
news_data = []

for entry in feed.entries:

    news_data.append({
        "company_name": entry.get("title"),
        "headline": entry.get("summary"),
        "published_at": entry.get("published"),
        "url": entry.get("link")
    })

nse_news = pd.DataFrame(news_data)

nse_news

,company_name,headline,published_at,url
0,Jubilant Pharmova Limited,Jubilant Pharmova Limited has informed the Exc...,23-Aug-2026 00:49:14,https://nsearchives.nseindia.com/corporate/JUB...
1,Pritish Nandy Communications Limited,Pursuant to Regulation 30 of the SEBI (Listing...,23-Aug-2026 00:00:07,https://nsearchives.nseindia.com/corporate/PNC...


In [35]:
print("Total announcements:", len(nse_news))
print("Columns:", nse_news.columns.tolist())

Total announcements: 2
Columns: ['company_name', 'headline', 'published_at', 'url']


In [36]:
nse_news["published_at"] = pd.to_datetime(
    nse_news["published_at"],
    format="%d-%b-%Y %H:%M:%S",
    errors="coerce"
)

nse_news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   company_name  2 non-null      object        
 1   headline      2 non-null      object        
 2   published_at  2 non-null      datetime64[ns]
 3   url           2 non-null      object        
dtypes: datetime64[ns](1), object(3)
memory usage: 196.0+ bytes


In [37]:
pd.set_option("display.max_colwidth", 150)

nse_news[
    ["company_name", "headline", "published_at"]
]

,company_name,headline,published_at
0,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",2026-08-23 00:49:14
1,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",2026-08-23 00:00:07


In [38]:
def fetch_nse_rss(url):

    response = requests.get(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
            ),
            "Accept": "application/rss+xml, application/xml, text/xml, */*"
        },
        timeout=30
    )

    response.raise_for_status()

    feed = feedparser.parse(response.content)

    news_data = []

    for entry in feed.entries:

        news_data.append({
            "company_name": entry.get("title"),
            "headline": entry.get("summary"),
            "published_at": entry.get("published"),
            "url": entry.get("link")
        })

    df = pd.DataFrame(news_data)

    if not df.empty:
        df["published_at"] = pd.to_datetime(
            df["published_at"],
            format="%d-%b-%Y %H:%M:%S",
            errors="coerce"
        )

    return df

In [39]:
online_url = (
    "https://nsearchives.nseindia.com/"
    "content/RSS/Online_announcements.xml"
)

test_news = fetch_nse_rss(online_url)

print("Announcements:", len(test_news))

test_news.head()

Announcements: 2


,company_name,headline,published_at,url
0,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",2026-08-23 00:49:14,https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf
1,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",2026-08-23 00:00:07,https://nsearchives.nseindia.com/corporate/PNC_22082026235955_Intimation_for_change_in_name_of_Company.pdf


In [40]:
results_url = (
    "https://nsearchives.nseindia.com/"
    "content/RSS/Financial_Results.xml"
)

results_news = fetch_nse_rss(results_url)

print("Financial results:", len(results_news))

results_news.head()

Financial results: 3


,company_name,headline,published_at,url
0,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,2026-08-06 14:07:24,https://archives.nseindia.com/corporate/xbrl/INDAS_121278_1708883_06082026020723.xml
1,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,2026-08-06 12:57:45,https://www.nseindia.com/companies-listing/corporate-filings-financial-results
2,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results


In [41]:
print("Online announcements:", len(test_news))
print("Financial results:", len(results_news))

Online announcements: 2
Financial results: 3


In [42]:
nse_news = pd.concat(
    [test_news, results_news],
    ignore_index=True
)

print("Total NSE records:", len(nse_news))

Total NSE records: 5


In [43]:
nse_feeds = {
    "online_announcement": 
        "https://nsearchives.nseindia.com/content/RSS/Online_announcements.xml",

    "financial_result":
        "https://nsearchives.nseindia.com/content/RSS/Financial_Results.xml",

    "board_meeting":
        "https://nsearchives.nseindia.com/content/RSS/Board_Meetings.xml",

    "corporate_action":
        "https://nsearchives.nseindia.com/content/RSS/Corporate_action.xml",

    "annual_report":
        "https://nsearchives.nseindia.com/content/RSS/Annual_Reports.xml"
}

In [44]:
def fetch_nse_rss(url, feed_type):

    response = requests.get(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 Chrome/150.0.0.0 Safari/537.36"
            ),
            "Accept": "application/rss+xml, application/xml, text/xml, */*"
        },
        timeout=30
    )

    response.raise_for_status()

    feed = feedparser.parse(response.content)

    news_data = []

    for entry in feed.entries:

        news_data.append({
            "company_name": entry.get("title"),
            "headline": entry.get("summary"),
            "published_at": entry.get("published"),
            "url": entry.get("link"),
            "source": "NSE",
            "feed_type": feed_type
        })

    df = pd.DataFrame(news_data)

    if not df.empty:
        df["published_at"] = pd.to_datetime(
            df["published_at"],
            format="%d-%b-%Y %H:%M:%S",
            errors="coerce"
        )

    return df

In [45]:
all_nse_news = []

for feed_type, url in nse_feeds.items():

    try:
        df = fetch_nse_rss(url, feed_type)

        print(
            f"{feed_type}: {len(df)} records"
        )

        all_nse_news.append(df)

    except Exception as e:

        print(
            f"{feed_type}: FAILED -> {e}"
        )

online_announcement: 2 records
financial_result: 3 records
board_meeting: 0 records
corporate_action: 77 records
annual_report: 20 records


In [46]:
all_nse_news = []

for feed_type, url in nse_feeds.items():

    try:
        df = fetch_nse_rss(url, feed_type)

        print(
            f"{feed_type}: {len(df)} records"
        )

        all_nse_news.append(df)

    except Exception as e:

        print(
            f"{feed_type}: FAILED -> {e}"
        )

online_announcement: 2 records
financial_result: 3 records
board_meeting: 0 records
corporate_action: 77 records
annual_report: 20 records


In [47]:
nse_news = pd.concat(
    all_nse_news,
    ignore_index=True
)

print("Total NSE records:", len(nse_news))


Total NSE records: 102


In [48]:
nse_news["feed_type"].value_counts()

feed_type
corporate_action       77
annual_report          20
financial_result        3
online_announcement     2
Name: count, dtype: int64

In [49]:
nse_news["company_name"].value_counts().head(20)


company_name
Mindspace Business Parks REIT                              2
Fonebox Retail Limited                                     2
Jubilant Pharmova Limited                                  1
GK Energy Limited - Ex-Date: 24-Aug-2026                   1
Amarjothi Spinning Mills Limited - Ex-Date: 21-Aug-2026    1
Alfred Herbert India Limited - Ex-Date: 21-Aug-2026        1
AK Capital Services Limited - Ex-Date: 21-Aug-2026         1
AIA Engineering Limited - Ex-Date: 04-Sep-2026             1
AGI Greenpac Limited - Ex-Date: 15-Sep-2026                1
GE Vernova T&D India Limited - Ex-Date: 21-Aug-2026        1
Gulshan Polyols Limited - Ex-Date: 28-Aug-2026             1
Goodluck India Limited - Ex-Date: 21-Aug-2026              1
DSP Mutual Fund - DSP Gold ETF - Ex-Date: 28-Aug-2026      1
Glenmark Pharmaceuticals Limited - Ex-Date: 31-Aug-2026    1
Gillette India Limited - Ex-Date: 24-Aug-2026              1
Arvind SmartSpaces Limited - Ex-Date: 28-Aug-2026          1
GANESH HOUS

In [50]:
print("Unique companies in news:", nse_news["company_name"].nunique())

Unique companies in news: 100


In [51]:
nse_news[
    ["company_name", "headline", "feed_type"]
].head(20)

,company_name,headline,feed_type
0,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",online_announcement
1,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",online_announcement
2,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,financial_result
3,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result
4,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result
5,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:INTERIM DIVIDEND - RS 2.55 PER SHARE |FACE VALUE:2 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
6,India Pesticides Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RE 0.75 PER SHARE |FACE VALUE:1 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
7,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 2 PER SHARE |FACE VALUE:1 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
8,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 3 PER SHARE |FACE VALUE:2 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action
9,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 4 PER SHARE |FACE VALUE:10 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action


In [52]:
import re

def clean_company_name(name):
    name = str(name).strip()

    # Corporate Action ke Ex-Date ko remove karo
    name = re.sub(
        r"\s*-\s*Ex-Date:\s*\d{2}-[A-Za-z]{3}-\d{4}",
        "",
        name,
        flags=re.IGNORECASE
    )

    return name.strip()

In [53]:
test_names = [
    "GK Energy Limited - Ex-Date: 24-Aug-2026",
    "JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026",
    "Jubilant Pharmova Limited"
]

for name in test_names:
    print(name, "→", clean_company_name(name))

GK Energy Limited - Ex-Date: 24-Aug-2026 → GK Energy Limited
JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026 → JINDAL STEEL LIMITED
Jubilant Pharmova Limited → Jubilant Pharmova Limited


In [54]:
nse_news["clean_company_name"] = (
    nse_news["company_name"]
    .apply(clean_company_name)
)

In [55]:
nse_news[
    ["company_name", "clean_company_name", "feed_type"]
].head(20)

,company_name,clean_company_name,feed_type
0,Jubilant Pharmova Limited,Jubilant Pharmova Limited,online_announcement
1,Pritish Nandy Communications Limited,Pritish Nandy Communications Limited,online_announcement
2,Kanani Industries Limited,Kanani Industries Limited,financial_result
3,Mindspace Business Parks REIT,Mindspace Business Parks REIT,financial_result
4,Mindspace Business Parks REIT,Mindspace Business Parks REIT,financial_result
5,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,International Gemological Institute Limited,corporate_action
6,India Pesticides Limited - Ex-Date: 24-Aug-2026,India Pesticides Limited,corporate_action
7,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,JINDAL STEEL LIMITED,corporate_action
8,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,Jindal Stainless Limited,corporate_action
9,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,Kalyani Forge Limited,corporate_action


In [56]:
def normalize_name(name):
    name = str(name).upper().strip()

    name = re.sub(r"\s+", " ", name)

    return name

In [57]:
nse_news["normalized_company_name"] = (
    nse_news["clean_company_name"]
    .apply(normalize_name)
)

In [58]:
nse_news[
    ["company_name", "clean_company_name"]
].drop_duplicates().head(30)

,company_name,clean_company_name
0,Jubilant Pharmova Limited,Jubilant Pharmova Limited
1,Pritish Nandy Communications Limited,Pritish Nandy Communications Limited
2,Kanani Industries Limited,Kanani Industries Limited
3,Mindspace Business Parks REIT,Mindspace Business Parks REIT
5,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,International Gemological Institute Limited
6,India Pesticides Limited - Ex-Date: 24-Aug-2026,India Pesticides Limited
7,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,JINDAL STEEL LIMITED
8,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,Jindal Stainless Limited
9,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,Kalyani Forge Limited
10,KSE Limited - Ex-Date: 21-Aug-2026,KSE Limited


In [59]:
print(
    "Unique raw names:",
    nse_news["company_name"].nunique()
)

print(
    "Unique cleaned names:",
    nse_news["clean_company_name"].nunique()
)

Unique raw names: 100
Unique cleaned names: 100


In [60]:
pip install mysql-connector-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 1.1 MB/s eta 0:00:00m eta 0:00:010:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [61]:
import mysql.connector

In [65]:
import os
from dotenv import load_dotenv
print(os.getcwd())

load_dotenv("../backend/.env")

print("DB Host:", os.getenv("DB_HOST"))
print("DB User:", os.getenv("DB_USER"))
print("DB Name:", os.getenv("DB_NAME"))

/Users/amit/Desktop/InvestIQ/ml
DB Host: localhost
DB User: root
DB Name: investiq


In [66]:
import mysql.connector

db = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

print("Database connected successfully")

Database connected successfully


In [67]:
query = """
SELECT
    id AS company_id,
    name AS db_company_name,
    symbol,
    isin
FROM companies
WHERE exchange = 'NSE';
"""

companies_df = pd.read_sql(query, db)

print("NSE companies:", len(companies_df))
companies_df.head()

NSE companies: 3117


/var/folders/vr/ysp63lq17s9ckkng6lrq12f40000gn/T/ipykernel_46573/2501147676.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  companies_df = pd.read_sql(query, db)


,company_id,db_company_name,symbol,isin
0,5,20 Microns Limited,20MICRONS,INE144J01027
1,6,21st Century Management Services Limited,21STCENMGM,INE253B01015
2,7,360 ONE WAM LIMITED,360ONE,INE466L01038
3,8,3B Blackbio Dx Limited,3BBLACKBIO,INE994E01018
4,9,3i Infotech Limited,3IINFOLTD,INE748C01038


In [68]:
companies_df["normalized_company_name"] = (
    companies_df["db_company_name"]
    .apply(normalize_name)
)

In [69]:
nse_news_mapped = nse_news.merge(
    companies_df[
        [
            "company_id",
            "db_company_name",
            "symbol",
            "isin",
            "normalized_company_name"
        ]
    ],
    on="normalized_company_name",
    how="left"
)

In [70]:
matched = nse_news_mapped["company_id"].notna().sum()
unmatched = nse_news_mapped["company_id"].isna().sum()

print("Matched:", matched)
print("Unmatched:", unmatched)
print(f"Mapping success: {matched / len(nse_news_mapped) * 100:.2f}%")

Matched: 97
Unmatched: 5
Mapping success: 95.10%


In [71]:
unmatched_news = nse_news_mapped[
    nse_news_mapped["company_id"].isna()
]

unmatched_news[
    ["company_name", "clean_company_name", "feed_type"]
].drop_duplicates()

,company_name,clean_company_name,feed_type
3,Mindspace Business Parks REIT,Mindspace Business Parks REIT,financial_result
31,Whirlpool of India Ltd - Ex-Date: 28-Aug-2026,Whirlpool of India Ltd,corporate_action
49,DSP Mutual Fund - DSP Silver ETF - Ex-Date: 28-Aug-2026,DSP Mutual Fund - DSP Silver ETF,corporate_action
66,DSP Mutual Fund - DSP Gold ETF - Ex-Date: 28-Aug-2026,DSP Mutual Fund - DSP Gold ETF,corporate_action


In [72]:
def normalize_name(name):
    name = str(name).upper().strip()

    name = re.sub(r"\s+", " ", name)

    # Common legal-name variations
    name = re.sub(r"\bLIMITED\b", "LTD", name)

    return name

In [73]:
nse_news["normalized_company_name"] = (
    nse_news["clean_company_name"]
    .apply(normalize_name)
)

companies_df["normalized_company_name"] = (
    companies_df["db_company_name"]
    .apply(normalize_name)
)

In [74]:
nse_news_mapped = nse_news.merge(
    companies_df[
        [
            "company_id",
            "db_company_name",
            "symbol",
            "isin",
            "normalized_company_name"
        ]
    ],
    on="normalized_company_name",
    how="left"
)

In [75]:
matched = nse_news_mapped["company_id"].notna().sum()
unmatched = nse_news_mapped["company_id"].isna().sum()

print("Matched:", matched)
print("Unmatched:", unmatched)
print(f"Mapping success: {matched / len(nse_news_mapped) * 100:.2f}%")

Matched: 98
Unmatched: 4
Mapping success: 96.08%


In [76]:
print("Total records:", len(nse_news_mapped))

duplicate_mask = nse_news_mapped.duplicated(
    subset=["company_id", "headline"],
    keep=False
)

duplicates = nse_news_mapped[duplicate_mask]

print("Duplicate records:", len(duplicates))

Total records: 102
Duplicate records: 4


In [77]:
duplicates[
    ["company_name", "headline", "feed_type"]
].sort_values("company_name")

,company_name,headline,feed_type
49,DSP Mutual Fund - DSP Silver ETF - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:FACE VALUE SPLIT (SUB-DIVISION) - FROM RS 10/- PER UNIT TO RE 1/- PER UNIT |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSUR...,corporate_action
66,DSP Mutual Fund - DSP Gold ETF - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:FACE VALUE SPLIT (SUB-DIVISION) - FROM RS 10/- PER UNIT TO RE 1/- PER UNIT |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSUR...,corporate_action
97,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report
99,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report


In [78]:
url_duplicates = nse_news_mapped[
    nse_news_mapped.duplicated(
        subset=["url"],
        keep=False
    )
]

print("URL duplicates:", len(url_duplicates))

URL duplicates: 79


In [79]:
before = len(nse_news_mapped)

nse_news_mapped = nse_news_mapped.drop_duplicates(
    subset=["company_id", "headline", "url"]
).reset_index(drop=True)

after = len(nse_news_mapped)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 102
After: 101
Removed: 1


In [80]:
nse_news_mapped[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url",
        "source"
    ]
].head()

,company_id,symbol,isin,company_name,headline,feed_type,published_at,url,source
0,1156.0,JUBLPHARMA,INE700A01033,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",online_announcement,2026-08-23 00:49:14,https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf,NSE
1,1723.0,PNC,INE392B01011,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",online_announcement,2026-08-23 00:00:07,https://nsearchives.nseindia.com/corporate/PNC_22082026235955_Intimation_for_change_in_name_of_Company.pdf,NSE
2,1177.0,KANANIIND,INE879E01037,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,financial_result,2026-08-06 14:07:24,https://archives.nseindia.com/corporate/xbrl/INDAS_121278_1708883_06082026020723.xml,NSE
3,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result,2026-08-06 12:57:45,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE
4,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE


In [81]:
url_counts = (
    nse_news_mapped["url"]
    .value_counts()
)

url_counts[url_counts > 1].head(20)

url
https://www.nseindia.com/companies-listing/corporate-filings-actions              76
https://www.nseindia.com/companies-listing/corporate-filings-financial-results     2
Name: count, dtype: int64

In [82]:
duplicate_urls = url_counts[url_counts > 1].index

nse_news_mapped[
    nse_news_mapped["url"].isin(duplicate_urls)
][
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url"
    ]
].sort_values("url")

,company_name,headline,feed_type,published_at,url
41,Protean eGov Technologies Limited - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 10 PER SHARE |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
58,Dynamatic Technologies Limited - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 5 PER SHARE |FACE VALUE:10 |RECORD DATE:28-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
57,DOMS Industries Limited - Ex-Date: 27-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 3.65 PER SHARE |FACE VALUE:10 |RECORD DATE:27-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
56,Diffusion Engineers Limited - Ex-Date: 25-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 1.50 PER SHARE |FACE VALUE:10 |RECORD DATE:26-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
55,Deep Industries Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 2.50 PER SHARE |FACE VALUE:5 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
...,...,...,...,...,...
25,Thomas Cook (India) Limited - Ex-Date: 27-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 0.50 PER SHARE |FACE VALUE:10 |RECORD DATE:27-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
24,TD Power Systems Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:FACE VALUE SPLIT (SUB-DIVISION) - FROM RS 2/- PER SHARE TO RE 1/- PER SHARE |FACE VALUE:2 |RECORD DATE:24-Aug-2026 |BOOK CLOSUR...,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
31,Whirlpool of India Ltd - Ex-Date: 28-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 5 PER SHARE |FACE VALUE: |RECORD DATE:28-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions
4,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results


In [83]:
nse_news_mapped.groupby("feed_type").agg(
    total_records=("feed_type", "size"),
    unique_urls=("url", "nunique"),
    unique_headlines=("headline", "nunique")
)

,total_records,unique_urls,unique_headlines
feed_type,,,
annual_report,20,20,2
corporate_action,76,1,72
financial_result,3,2,3
online_announcement,2,2,2


In [84]:
nse_news_mapped["headline_normalized"] = (
    nse_news_mapped["headline"]
    .fillna("")
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [85]:
duplicate_mask = nse_news_mapped.duplicated(
    subset=[
        "company_id",
        "headline_normalized",
        "feed_type"
    ],
    keep=False
)

smart_duplicates = nse_news_mapped[duplicate_mask]

print("Potential smart duplicates:", len(smart_duplicates))

Potential smart duplicates: 2


In [86]:
smart_duplicates[
    [
        "company_name",
        "headline",
        "feed_type",
        "published_at"
    ]
].sort_values(
    ["company_name", "feed_type", "published_at"]
)

,company_name,headline,feed_type,published_at
96,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT
98,Fonebox Retail Limited,AS ON DATE : 21-AUG-26,annual_report,NaT


## Finalized Duplication 

In [87]:
before = len(nse_news_mapped)

nse_news_mapped = nse_news_mapped.drop_duplicates(
    subset=[
        "company_id",
        "headline_normalized",
        "feed_type"
    ],
    keep="first"
).reset_index(drop=True)

after = len(nse_news_mapped)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

Before: 101
After: 100
Removed: 1


In [88]:
nse_news_mapped[
    [
        "company_id",
        "symbol",
        "isin",
        "company_name",
        "headline",
        "feed_type",
        "published_at",
        "url",
        "source"
    ]
].head(10)

,company_id,symbol,isin,company_name,headline,feed_type,published_at,url,source
0,1156.0,JUBLPHARMA,INE700A01033,Jubilant Pharmova Limited,"Jubilant Pharmova Limited has informed the Exchange about General Updates-USFDA Approval for Line 3, Contract manufacturing facility, Spokane, U.S...",online_announcement,2026-08-23 00:49:14,https://nsearchives.nseindia.com/corporate/JUBLPHARMA_23082026004902_USFDA_Intimation_August23.pdf,NSE
1,1723.0,PNC,INE392B01011,Pritish Nandy Communications Limited,"Pursuant to Regulation 30 of the SEBI (Listing Obligations and Disclosure Requirements) Regulations, 2015, we are pleased to inform that the compa...",online_announcement,2026-08-23 00:00:07,https://nsearchives.nseindia.com/corporate/PNC_22082026235955_Intimation_for_change_in_name_of_Company.pdf,NSE
2,1177.0,KANANIIND,INE879E01037,Kanani Industries Limited,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:Non-Cumulative |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |...,financial_result,2026-08-06 14:07:24,https://archives.nseindia.com/corporate/xbrl/INDAS_121278_1708883_06082026020723.xml,NSE
3,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Consolidated |IND AS/ NON IND A...,financial_result,2026-08-06 12:57:45,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE
4,NaN,NaN,NaN,Mindspace Business Parks REIT,RELATING TO:First Quarter |AUDITED/UNAUDITED:Unaudited |CUMULATIVE/NON-CUMULATIVE:- |CONSOLIDATED/NON-CONSOLIDATED:Non-Consolidated |IND AS/ NON I...,financial_result,2026-08-06 12:55:40,https://www.nseindia.com/companies-listing/corporate-filings-financial-results,NSE
5,1002.0,IGIL,INE0Q9301021,International Gemological Institute Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:INTERIM DIVIDEND - RS 2.55 PER SHARE |FACE VALUE:2 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
6,1073.0,IPL,INE0D6701023,India Pesticides Limited - Ex-Date: 24-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RE 0.75 PER SHARE |FACE VALUE:1 |RECORD DATE:24-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
7,1121.0,JINDALSTEL,INE749A01030,JINDAL STEEL LIMITED - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 2 PER SHARE |FACE VALUE:1 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
8,1143.0,JSL,INE220G01021,Jindal Stainless Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 3 PER SHARE |FACE VALUE:2 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE
9,1170.0,KALYANIFRG,INE314G01014,Kalyani Forge Limited - Ex-Date: 21-Aug-2026,SERIES:EQ |PURPOSE:DIVIDEND - RS 4 PER SHARE |FACE VALUE:10 |RECORD DATE:21-Aug-2026 |BOOK CLOSURE START DATE:- |BOOK CLOSURE END DATE:-,corporate_action,2026-08-21 06:08:44,https://www.nseindia.com/companies-listing/corporate-filings-actions,NSE


In [89]:
print("Final NSE news records:", len(nse_news_mapped))
print(
    nse_news_mapped["feed_type"].value_counts()
)

Final NSE news records: 100
feed_type
corporate_action       76
annual_report          19
financial_result        3
online_announcement     2
Name: count, dtype: int64


In [ ]:
#Robust timestamps.

In [90]:
nse_news_mapped["collected_at"] = pd.Timestamp.now()

In [91]:
nse_news_mapped[
    ["company_name", "published_at", "collected_at"]
].head()

,company_name,published_at,collected_at
0,Jubilant Pharmova Limited,2026-08-23 00:49:14,2026-08-23 06:58:34.770830
1,Pritish Nandy Communications Limited,2026-08-23 00:00:07,2026-08-23 06:58:34.770830
2,Kanani Industries Limited,2026-08-06 14:07:24,2026-08-23 06:58:34.770830
3,Mindspace Business Parks REIT,2026-08-06 12:57:45,2026-08-23 06:58:34.770830
4,Mindspace Business Parks REIT,2026-08-06 12:55:40,2026-08-23 06:58:34.770830


In [92]:
print("Total records:", len(nse_news_mapped))
print("Valid published_at:", nse_news_mapped["published_at"].notna().sum())
print("Missing published_at:", nse_news_mapped["published_at"].isna().sum())

Total records: 100
Valid published_at: 81
Missing published_at: 19


In [93]:
missing_dates = nse_news_mapped[
    nse_news_mapped["published_at"].isna()
]

missing_dates[
    [
        "company_name",
        "headline",
        "feed_type",
        "url"
    ]
]

,company_name,headline,feed_type,url
81,ABS Marine Services Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30522_ABSMARINE_2025_2026_A_4942091_22082026195020.pdf
82,Gujarat Narmada Valley Fertilizers and Chemicals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30521_GNFC_2025_2026_A_17228642_22082026180508.pdf
83,Sp Refractories Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30520_SPRL_2025_2026_A_5465386_22082026173709.pdf
84,Infinium Pharmachem Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30519_INFINIUM_2025_2026_A_2216015_22082026172907.pdf
85,Bannari Amman Sugars Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30518_BANARISUG_2025_2026_A_59448263_22082026165031.pdf
86,Laxmi India Finance Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30517_LAXMIINDIA_2025_2026_A_11673486_22082026155431.pdf
87,Astra Microwave Products Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30516_ASTRAMICRO_2025_2026_A_8389931_22082026143020.pdf
88,Eastern Silk Industries Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30515_EASTSILK_2025_2026_U_6588226_22082026142034.pdf
89,MIC Electronics Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30514_MICEL_2025_2026_A_2371653_22082026125810.pdf
90,Shri Ahimsa Naturals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30513_SHRIAHIMSA_2025_2026_A_5998714_22082026120521.pdf


In [95]:
nse_news_mapped["published_at"] = pd.to_datetime(
    nse_news_mapped["published_at"],
    errors="coerce"
)

In [96]:
print(nse_news_mapped["published_at"].dtype)

datetime64[ns]


In [97]:
print("Oldest:", nse_news_mapped["published_at"].min())
print("Newest:", nse_news_mapped["published_at"].max())

Oldest: 2026-08-06 12:55:40
Newest: 2026-08-23 00:49:14


In [98]:
print("Total records:", len(nse_news_mapped))
print("Valid published_at:", nse_news_mapped["published_at"].notna().sum())
print("Missing published_at:", nse_news_mapped["published_at"].isna().sum())

Total records: 100
Valid published_at: 81
Missing published_at: 19


In [99]:
missing_dates = nse_news_mapped[
    nse_news_mapped["published_at"].isna()
]

missing_dates[
    ["company_name", "headline", "feed_type", "url"]
]

,company_name,headline,feed_type,url
81,ABS Marine Services Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30522_ABSMARINE_2025_2026_A_4942091_22082026195020.pdf
82,Gujarat Narmada Valley Fertilizers and Chemicals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30521_GNFC_2025_2026_A_17228642_22082026180508.pdf
83,Sp Refractories Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30520_SPRL_2025_2026_A_5465386_22082026173709.pdf
84,Infinium Pharmachem Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30519_INFINIUM_2025_2026_A_2216015_22082026172907.pdf
85,Bannari Amman Sugars Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30518_BANARISUG_2025_2026_A_59448263_22082026165031.pdf
86,Laxmi India Finance Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30517_LAXMIINDIA_2025_2026_A_11673486_22082026155431.pdf
87,Astra Microwave Products Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30516_ASTRAMICRO_2025_2026_A_8389931_22082026143020.pdf
88,Eastern Silk Industries Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30515_EASTSILK_2025_2026_U_6588226_22082026142034.pdf
89,MIC Electronics Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/AR_30514_MICEL_2025_2026_A_2371653_22082026125810.pdf
90,Shri Ahimsa Naturals Limited,AS ON DATE : 22-AUG-26,annual_report,https://nsearchives.nseindia.com/annual_reports/SME_AR_30513_SHRIAHIMSA_2025_2026_A_5998714_22082026120521.pdf


In [100]:
nse_news_mapped["published_at_original"] = nse_news_mapped["published_at"]

In [101]:
import re
import pandas as pd

def extract_date_from_headline(row):
    if pd.notna(row["published_at"]):
        return row["published_at"]

    headline = str(row["headline"])

    match = re.search(
        r"AS ON DATE\s*:\s*(\d{2}-[A-Z]{3}-\d{2})",
        headline,
        re.IGNORECASE
    )

    if match:
        return pd.to_datetime(
            match.group(1),
            format="%d-%b-%y",
            errors="coerce"
        )

    return pd.NaT

In [102]:
nse_news_mapped["published_at"] = (
    nse_news_mapped.apply(
        extract_date_from_headline,
        axis=1
    )
)

In [103]:
print(
    "Missing published_at:",
    nse_news_mapped["published_at"].isna().sum()
)

Missing published_at: 0


In [104]:
nse_news_mapped[
    nse_news_mapped["feed_type"] == "annual_report"
][
    ["company_name", "headline", "published_at"]
].head(20)

,company_name,headline,published_at
81,ABS Marine Services Limited,AS ON DATE : 22-AUG-26,2026-08-22
82,Gujarat Narmada Valley Fertilizers and Chemicals Limited,AS ON DATE : 22-AUG-26,2026-08-22
83,Sp Refractories Limited,AS ON DATE : 22-AUG-26,2026-08-22
84,Infinium Pharmachem Limited,AS ON DATE : 22-AUG-26,2026-08-22
85,Bannari Amman Sugars Limited,AS ON DATE : 22-AUG-26,2026-08-22
86,Laxmi India Finance Limited,AS ON DATE : 22-AUG-26,2026-08-22
87,Astra Microwave Products Limited,AS ON DATE : 22-AUG-26,2026-08-22
88,Eastern Silk Industries Limited,AS ON DATE : 22-AUG-26,2026-08-22
89,MIC Electronics Limited,AS ON DATE : 22-AUG-26,2026-08-22
90,Shri Ahimsa Naturals Limited,AS ON DATE : 22-AUG-26,2026-08-22
